# Supplementary Figure 5: error controls at different selection stages

Panels a-f show what each error control measures: the fusion-linked diagnostic in four cohorts (a-c), the explicit-peptide donor and self-selected entrapment controls (d) and the known-truth simulations (e,f). The continuation page decomposes the CAMPI fusion diagnostic into paired and unresolved parts. Every panel is redrawn from `source_data/SFig05/` with the v30 plotting code and compared pixel by pixel with the v30 render.

Run from the repository root: `python paper.py build SFig05`.

In [ ]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # rendered panels; gitignored
OUT.mkdir(parents=True, exist_ok=True)
from style import fl_style as fl
import panels
fl.init()
import pandas as pd
from IPython.display import Image, display
T = panels.load(SD)

## Panels a-f

Panels a-f are drawn as one figure (`panels.figure_a_to_f`, which calls `panel_ab`, `panel_c`, `panel_d` and `panel_ef`), saved as PDF with `fl_style.save` and rasterised at 300 dpi, as in the v30 build.

- a,b: fusion-linked diagnostic at peptide q≤0.01 for 5-50 and 9-50 residues, split into paired and unresolved parts; one bar per cohort and seed. Unit: acquisition counts summed within cohort and seed (CAMPI 15, healthy stool 58, PREDICT 306, CapScan 331 acquisitions; two seeds). Descriptive, no test.
- c: nested 5-8-residue population, one dot per seed, bar at the median of the two seeds. Descriptive.
- d: explicit-peptide entrapment, self-selected (5 acquisitions) and donor-selected (58 acquisitions), each acquisition the mean of three seeds. Supports "The median entrapment estimate was 1.0% for accepted 5–50-residue peptides and 0.49% for the 9–50-residue subset" (visible_v30f line 30). Descriptive.
- e,f: realised false-discovery proportion in 12 baseline and 12 stress mixtures for the fixed, independent and self arms; squares are means with 95% whole-mixture percentile bootstrap intervals (20,000 resamples). Supports "did not increase the false-discovery proportion under baseline conditions (1.01% versus 1.25% with a fixed reference)" and "the proportion increased to 1.91%, 0.71 percentage points above selection from an independent replicate" (visible_v30f line 46). Unit: synthetic mixture, n = 12 per regime.

In [ ]:
png = panels.figure_a_to_f(T, OUT)
display(Image(filename=str(png), width=700))

### Panels a-c: pooled values recomputed from acquisition components

Summing acquisition-level numerators and denominators within cohort and seed (Methods, visible_v30f line 367) reproduces every drawn value.

In [ ]:
pooled = panels.stat_ac_fusion_pooled(T["fusion_acquisition_components"])
display(panels.compare_pooled(pooled, T["fusion_pooled_current"]))
display(pooled[pooled.q_threshold.eq(panels.Q_CUTOFF) & pooled.max_residues.eq(50)].groupby("cohort").acquisitions.first())

### Panel d: seed means and medians recomputed from per-seed records

In [ ]:
seed_means = panels.stat_d_donor_seed_means(T["donor_recount"])
display(panels.compare_frames(seed_means, T["donor_entrapment_seed_means"], ["target", "donor", "tier", "group", "min_length", "q_threshold"]))
display(panels.stat_d_donor_summary(seed_means))

### Panels e,f: bootstrap intervals recomputed

The bootstrap is rerun with the producer's generator, seed (2026091051) and draw order on all 288 simulation outcomes, of which the 72 ordinary outcomes are used.

In [ ]:
unc = panels.stat_ef_simulation_uncertainty(T["simulation_all_outcomes"])
display(panels.compare_frames(unc, T["simulation_uncertainty"], ["regime", "endpoint"]))
print("max |FDP - false/max(discoveries,1)| over 72 outcomes:", panels.stat_ef_fdp_identity(T["simulation_mixture_outcomes"]))
display(unc)

## Panel continued

Decomposition of the CAMPI fusion diagnostic: 15 acquisitions pooled separately for each seed from the 375,333,654-sequence reservoir at peptide q≤0.01. Blue is the weighted paired contribution (E + E₀ + 2B)/N, grey the unresolved contribution U/N; labels give the total. Supports Methods visible_v30f line 367 ("the pooled diagnostic was 2.59–2.79% for 5–50 residues and 1.55–1.56% for the nested 9–50-residue subset") and line 45 ("the pooled diagnostic averaged 1.6% for 9–50-residue peptides over two randomizations"). Unit: acquisition counts summed within seed (n = 15 per seed, two seeds). Descriptive, no test.

In [ ]:
png = panels.figure_continued(T, OUT)
display(Image(filename=str(png), width=700))
dec = panels.stat_cont_fusion_decomposition(T["fusion_acquisition_components"])
display(panels.compare_frames(dec.drop(columns="acquisitions"), T["fusion_decomposition_current375"], ["seed", "min", "max"]))
display(dec)

## Checks and numbers

In [ ]:
import os
from tools.check_pixels import compare
if os.environ.get("FASTALAKE_FONT_DIR") or os.environ.get("FL_STYLE_DIR"):
    for name, ref in [("SFig05", "SFig05_v42"), ("SFig05_continued", "SFig05_continued_v35")]:   # a-f page re-labelled "Realized" in v33, cohort names in v35; continued page integer labels in v35
        r = compare(OUT / f"{name}.png", UNIT / "expected" / f"{ref}.png", UNIT / "checks" / ref, dpi=300)
        print(name, r["verdict"], r["differing_pixels"])
        assert r["verdict"] == "IDENTICAL", r
else:
    print("pixel checks SKIPPED: fonts absent")
for key, value in panels.numbers(T):
    print(f"{key}\t{value!r}")